In [ ]:
"""
Sample LangChain script: Chain-of‑Thought scoring of Caldara (2018) 8‑class Geopolitical Risks
from 10‑K Item 1A (Risk Factors) and Item 7 (MD&A).

Author: ChatGPT
Date: 2025‑07‑04
"""
from __future__ import annotations

import os
import json
from pathlib import Path
from typing import List, Dict

from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import BaseOutputParser

# ---------------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------------
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL_NAME = "gpt-4o-mini"  # adjust to the deployed model id

# Caldara & Iacoviello (2018) 8‑class taxonomy ------------------------------
RISK_CATEGORIES = [
    "War threats",        # Threat of conventional military conflict
    "Peace threats",      # Breakdown of diplomatic efforts / threats to peace
    "Military buildups",  # Significant troop or arms build‑up
    "Nuclear threats",    # Nuclear weapon development or deployment risk
    "Terror threats",     # Threats or warnings of terrorist activity
    "Beginning of war",   # Formal outbreak or declaration of war
    "Escalation of war",  # Intensified or widening conflict
    "Terror acts"         # Actual terrorist attacks / hostage situations
]

SCORE_SCALE = "0 (none) – 5 (very severe)"

# ---------------------------------------------------------------------------
# 2. Prompt Template with explicit Chain‑of‑Thought
# ---------------------------------------------------------------------------
PROMPT_TMPL = """
You are a senior geopolitical risk analyst. Carefully read the following excerpt from a company's annual report (10‑K). For each risk category listed below, **think step‑by‑step** about whether the text contains evidence of that type of risk in the firm's business environment **in the given financial year**. Then assign a score on a {scale} scale.

*Return your reasoning for each category, followed by a JSON object named `scores` containing only the final numeric scores.*

### Risk categories
{categories}

### 10‑K excerpt
"""

prompt = PromptTemplate(
    template=PROMPT_TMPL + """{context}\n""",
    input_variables=["context"],
    partial_variables={
        "categories": "\n".join(f"- {c}" for c in RISK_CATEGORIES),
        "scale": SCORE_SCALE,
    },
)

# ---------------------------------------------------------------------------
# 3. Optional: JSON parser to extract the machine‑readable result -------------
# ---------------------------------------------------------------------------
class ScoreParser(BaseOutputParser):
    """Extract the JSON `scores` object from the Chat model output."""

    def parse(self, text: str) -> Dict[str, int]:
        try:
            start = text.index("{")
            end = text.rindex("}") + 1
            return json.loads(text[start:end])["scores"]
        except Exception as exc:
            raise ValueError("Failed to parse scores JSON") from exc

# ---------------------------------------------------------------------------
# 4. Utility: load and chunk Items 1A & 7 -----------------------------------
# ---------------------------------------------------------------------------

def load_items(item1a_path: str | Path, item7_path: str | Path) -> str:
    """Concatenate Item 1A and Item 7 text."""
    text1a = Path(item1a_path).read_text(encoding="utf-8")
    text7 = Path(item7_path).read_text(encoding="utf-8")
    return text1a + "\n\n" + text7


def chunk_text(text: str, chunk_size: int = 4000, chunk_overlap: int = 200) -> List[str]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(text)

# ---------------------------------------------------------------------------
# 5. Main scoring function ---------------------------------------------------
# ---------------------------------------------------------------------------

def score_gpr(item1a_path: str | Path, item7_path: str | Path) -> Dict[str, int]:
    """Return GPR scores (0‑5) for the given 10‑K sections."""
    raw_text = load_items(item1a_path, item7_path)
    chunks = chunk_text(raw_text)

    llm = ChatOpenAI(model_name=MODEL_NAME, temperature=0.2)
    parser = ScoreParser()
    chain = LLMChain(llm=llm, prompt=prompt, output_parser=parser)

    # --- Aggregate evidence from all chunks (take max score per category) ---
    aggregated = {cat: 0 for cat in RISK_CATEGORIES}

    for chunk in chunks:
        result: Dict[str, int] = chain.run(context=chunk)
        for cat, val in result.items():
            aggregated[cat] = max(aggregated[cat], val)

    return aggregated

# ---------------------------------------------------------------------------
# 6. Example CLI usage -------------------------------------------------------
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Score GPR risks from 10‑K Items 1A & 7")
    parser.add_argument("item1a", type=Path, help="Path to extracted Item 1A text file")
    parser.add_argument("item7", type=Path, help="Path to extracted Item 7 text file")
    args = parser.parse_args()

    scores = score_gpr(args.item1a, args.item7)
    print(json.dumps(scores, indent=2, ensure_ascii=False))
